# Module project: Computational Chemistry with Python

### Group L : Taïs Thomas, Marouchka Heck, Marie-Caroline Bilocq

## PART A - Data structures and functions

### 1. Load the "periodic_table.csv" data file using pandas

We first load the periodic data table using pandas and preview the first rows to check its structure and column names, which will be useful for Question 2 when we need to call the columns for the dictionnary.

In [60]:
import pandas as pd

periodic_table = pd.read_csv("periodic_table.csv")
periodic_table.head(5)

,AtomicNumber,Symbol,Name,AtomicMass,CPKHexColor,ElectronConfiguration,Electronegativity,AtomicRadius,IonizationEnergy,ElectronAffinity,OxidationStates,StandardState,MeltingPoint,BoilingPoint,Density,GroupBlock,YearDiscovered
0,1,H,Hydrogen,1.008000,FFFFFF,1s1,2.20,120.0,13.598,0.754,"+1, -1",Gas,13.81,20.28,0.000090,Nonmetal,1766
1,2,He,Helium,4.002600,D9FFFF,1s2,NaN,140.0,24.587,NaN,0,Gas,0.95,4.22,0.000179,Noble gas,1868
2,3,Li,Lithium,7.000000,CC80FF,[He]2s1,0.98,182.0,5.392,0.618,+1,Solid,453.65,1615.00,0.534000,Alkali metal,1817
3,4,Be,Beryllium,9.012183,C2FF00,[He]2s2,1.57,153.0,9.323,NaN,+2,Solid,1560.00,2744.00,1.850000,Alkaline earth metal,1798
4,5,B,Boron,10.810000,FFB5B5,[He]2s2 2p1,2.04,192.0,8.298,0.277,+3,Solid,2348.00,4273.00,2.370000,Metalloid,1808


### 2. Create a python dictionnary where the keys are elements symbol and the values are atomic masses.

In [61]:
atomic_masses = dict(zip(periodic_table["Symbol"], periodic_table["AtomicMass"]))

### 3. Write a function that takes chemical formulas like “H₂O” and “C₆H₁₂O₆” and return their molecular mass.

I'll iterate through every character in the formula and check whether it's a letter or a number. Then I will assign the corresponding atomic mass if it is a letter and if it is a number I will multiply the previous element atomic mass by that number.

In [62]:
"""""
def molecular_mass(chemical_formula):
    mass = 0
    
    for character in chemical_formula:
        if character.isalpha():
            element = character
            mass += atomic_masses[element]
        elif character.isdigit():
            number = int(character)
            mass += atomic_masses[element] * (int(character) - 1)

    return mass

print(molecular_mass("H2O"))
print(molecular_mass("C6H12O6"))
"""

'""\ndef molecular_mass(chemical_formula):\n    mass = 0\n    \n    for character in chemical_formula:\n        if character.isalpha():\n            element = character\n            mass += atomic_masses[element]\n        elif character.isdigit():\n            number = int(character)\n            mass += atomic_masses[element] * (int(character) - 1)\n\n    return mass\n\nprint(molecular_mass("H2O"))\nprint(molecular_mass("C6H12O6"))\n'

I first tried to read the formula character by character and separate the letters from the numbers. This works for a simple formula like $\mathrm{H_2O}$, but it does not work correctly for $\mathrm{C_6H_{12}O_6}$, because the 12 is read as 1 and 2 separately instead of as the number 12. Another problem would appear with elements that have two-letter symbols, such as Ca or Cl, since the code would read each letter as a different element. To solve these problems, I used a regular expression `re.findall` to identify the complete element symbol together with the number that follows it. If there is no number after an element, the number of atoms is set to 1. This allows the molecular mass to be calculated correctly by multiplying each atomic mass by the corresponding number of atoms.

#### Corrected version of question 3

In [63]:
import re

def molecular_mass(chemical_formula):
    mass = 0
    elements = re.findall(r"([A-Z][a-z]?)(\d*)", chemical_formula)

    for element, number in elements:
        if number == "":
            number = 1
        else:
            number = int(number)

        mass += atomic_masses[element] * number

    return mass

print(molecular_mass("H2O"))
print(molecular_mass("C6H12O6"))

18.015
180.156


### 4. Extend the molecular mass calculator to handle parentheses in formulas (e.g., “Ca(OH)₂”).

To extend the function from Question 3, I redefine `molecular_mass()` to first check for parentheses. If there are no parentheses, the calculation remains the same as in Question 3.

In [64]:
def molecular_mass(chemical_formula):
    mass = 0

    if "(" in chemical_formula:
        start = chemical_formula.find("(")
        end = chemical_formula.find(")")

        group = chemical_formula[start + 1:end]

        position = end + 1
        number = ""

        while position < len(chemical_formula) and chemical_formula[position].isdigit():
            number += chemical_formula[position]
            position += 1

        if number == "":
            number = 1
        else:
            number = int(number)

        group_mass = molecular_mass(group)

        remaining_formula = chemical_formula[:start] + chemical_formula[position:]

        return molecular_mass(remaining_formula) + group_mass * number

    elements = re.findall(r"([A-Z][a-z]?)(\d*)", chemical_formula)

    for element, number in elements:
        if number == "":
            number = 1
        else:
            number = int(number)

        mass += atomic_masses[element] * number

    return mass

print(molecular_mass("Ca(OH)2"))

74.094


To handle parentheses, I first identify the part of the formula inside the parentheses and the number that follows it. I then use recursion by calling `molecular_mass()` again on the group inside the parentheses. The mass of this group is multiplied by the number after the parentheses and added to the mass of the remaining formula. If there are no parentheses, the function uses the same method as in Question 3.

### 5. Extend the molecular mass calculator so it can handle coordinated (hydration) water (e.g. “CuSO4.5H2O”) (5 points)

To extend the function from Question 4, I redefine `molecular_mass()` to first check for hydration water. If there is no hydration water, the calculation remains the same as in Question 4.

In [65]:
def molecular_mass(chemical_formula):
    mass = 0

    if "." in chemical_formula:
        parts = chemical_formula.split(".")

        compound = parts[0]
        hydrate = parts[1]

        match = re.match(r"(\d+)(.*)", hydrate)
        number = int(match.group(1))
        water = match.group(2)

        return molecular_mass(compound) + number * molecular_mass(water)

    if "(" in chemical_formula:
        start = chemical_formula.find("(")
        end = chemical_formula.find(")")

        group = chemical_formula[start + 1:end]

        position = end + 1
        number = ""

        while position < len(chemical_formula) and chemical_formula[position].isdigit():
            number += chemical_formula[position]
            position += 1

        if number == "":
            number = 1
        else:
            number = int(number)

        group_mass = molecular_mass(group)

        remaining_formula = chemical_formula[:start] + chemical_formula[position:]

        return molecular_mass(remaining_formula) + group_mass * number

    elements = re.findall(r"([A-Z][a-z]?)(\d*)", chemical_formula)

    for element, number in elements:
        if number == "":
            number = 1
        else:
            number = int(number)

        mass += atomic_masses[element] * number

    return mass

print(molecular_mass("CuSO4.5H2O"))

249.69100000000003


To extend the function to hydration water, I first check if the formula contains a ".". If it does, I split the formula into the main compound and the hydrate part. I then separate the coefficient from the water formula and calculate the molecular mass of both parts using the `molecular_mass()` function. Finally, the mass of the water is multiplied by its coefficient and added to the mass of the main compound.

## PART B - Stoichiometry and reaction balancing

### 1. Reaction Balancer Function

### 2. Mass Conservation Check

## PART C - Simulation / Modeling